In [ ]:
import jax.numpy as np
import jax.random as random
import matplotlib.pyplot as plt
import seaborn as sns

n = 10_000_000

key = random.PRNGKey(0)

original_sample = random.normal(key, shape=(n,))
print("done generating samples")
data = original_sample * 0.5 + 0.5
data2 = data + 0.1

data_bounded = np.clip(data, -1, 1)
data_bounded2 = np.clip(data2, -1, 1)

print("done clipping")

data_tanh = np.tanh(data)
data_tanh2 = np.tanh(data2)

print("done bounding samples")

plt.figure(figsize=(10, 6))
plt.hist(np.array(data), bins=100, density=True, alpha=0.5, label='Unbounded Gaussian (μ=0, σ=0.5)')

print("computing histograms")

# Convert jax arrays to numpy arrays before plotting with seaborn
sns.kdeplot(np.array(data_bounded), bw_adjust=0.5, fill=False, alpha=0.5, label='Clipped Gaussian to [-1,1]', color='orange')
sns.kdeplot(np.array(data_tanh), bw_adjust=0.5, fill=False, alpha=0.5, label='Bounded Gaussian (tanh)', color='green')

plt.xlabel('Value', fontsize=22)
plt.ylabel('Density', fontsize=22)
plt.title('Histograms of Gaussian and Bounded Gaussian', fontsize=22)
plt.legend(fontsize=16)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.savefig("./plots/bound/clipped_gaussian.pdf", format="pdf", bbox_inches="tight")
plt.show()


done generating samples
done clipping
done bounding samples


In [11]:
def compute_clipping_ratios(data,data2):

    # Use common bins for both histograms
    bins = np.linspace(min(np.min(data), np.min(data2)),
                    max(np.max(data), np.max(data2)),
                    20)

    # Calculate density histograms
    h1, _ = np.histogram(data, bins=bins, density=True)
    h2, _ = np.histogram(data2, bins=bins, density=True)
    bin_centers = (bins[:-1] + bins[1:]) / 2

    # Compute the ratio (handle division by zero)
    ratio = np.divide(h1, h2, out=np.full_like(h1, np.nan), where=h2>1e-10)

    # Identify the maximum ratio
    max_ratio = np.nanmax(ratio)
    max_index = np.nanargmax(ratio)
    median_ratio = np.nanmedian(ratio)

    # # Plot the ratio vs bin centers and mark the maximum ratio
    # plt.figure(figsize=(10, 6))
    # plt.bar(bin_centers, ratio, width=bins[1]-bins[0], alpha=0.5, label='Probability Ratio (data/data2)')
    # plt.axhline(max_ratio, color='red', linestyle='--', 
    #             label=f'Max Ratio = {max_ratio:.2f} at {bin_centers[max_index]:.2f}')
    # plt.axhline(median_ratio, color='green', linestyle='--',
    #             label=f'Median Ratio = {median_ratio:.2f}')
    # plt.xlabel('Value')
    # plt.ylabel('Probability Ratio')
    # plt.title('Max Ratio Disparity between data and data2')
    # plt.legend()
    # plt.show()
    
    return max_ratio, median_ratio


def compute_kl_divergence(data,data2,plot=False):

    bins = np.linspace(min(np.min(data), np.min(data2)),
                    max(np.max(data), np.max(data2)),
                    50)

    # Calculate density histograms
    h1, _ = np.histogram(data, bins=bins, density=True)
    h2, _ = np.histogram(data2, bins=bins, density=True)
    bin_centers = (bins[:-1] + bins[1:]) / 2

    kl_divergence = np.sum(h1 * np.log(h1/h2))

    if plot:
        # Plot the density histograms
        plt.figure(figsize=(10, 6))
        plt.bar(bin_centers, h1, width=bins[1]-bins[0], alpha=0.5, label='Data 1')
        plt.bar(bin_centers, h2, width=bins[1]-bins[0], alpha=0.5, label='Data 2')
        plt.xlabel('Value')
        plt.ylabel('Density')
        plt.title(f'Mean = {data.mean()} std = {data2.std()} and Data 2 KL-Divergence = {kl_divergence:.2f}')
        plt.legend()
        plt.show()
    return bin_centers,h1,h2,kl_divergence



#compute_clipping_ratios(data_bounded,data_bounded2)
rslt = compute_kl_divergence(data_bounded,data_bounded2)


In [ ]:
import seaborn as sns
import itertools

sns.set(font_scale=10)
sns.set_theme()


ratios = []
means = np.linspace(-1, 1., 201)
stds = np.linspace(0.1,1., 201)[::-1]

list = [0.,0.4,0.8]

bins,hist1 , hist2 = [], [], []


for mean,_ in zip(means[:-10],stds[:-10]): 

    #print(mean)

    std = 0.2
    #std = 0.5
    
    data = original_sample * std + mean
    #data2 = np.random.normal(loc=np.clip(mean+0.1,max=1.), scale=std, size=n)
    data2 = data + 0.1
    data_bounded = np.clip(data, -1, 1)
    data_bounded2 = np.clip(data2, -1, 1)
    
    #max_ratio, median_ratio = compute_clipping_ratios(data_bounded,data_bounded2)
    #bin,h1,h2,max_ratio = compute_kl_divergence(data_bounded,data_bounded2,plot= mean in list)
    bin,h1,h2,max_ratio = compute_kl_divergence(data_bounded,data_bounded2,plot= False)
    ratios.append(max_ratio)

    if round(mean, 3) in list:

        print(f'RATIOOOOOOOOOOOO {mean} {max_ratio}')
        hist1.append(h1)
        hist2.append(h2)
        bins.append(bin)





# plt.yticks(fontsize=16)

# plt.xlabel('Mean of the Gaussian',fontsize=22)
# plt.ylabel('Max Ratio',fontsize=22)

# Plot the histograms h1 and h2 on the same figure
import matplotlib.cm as cm

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot the histograms h1 and h2 on the same figure
num_curves = len(bins)
for i, (bin_centers, h1_hist, h2_hist) in enumerate(zip(bins, hist1, hist2)):
    # Normalize i to [0,1] to vary the color from light to dark
    norm = i / (num_curves - 1) if num_curves > 1 else 0.5
    # Adjust the range in the colormap, here using 0.3 to 1 for a better visible range
    blue_color = cm.Blues(0.3 + 0.7 * norm)
    orange_color = cm.Oranges(0.3 + 0.7 * norm)
    
    ax1.plot(bin_centers, h1_hist, color=blue_color, lw=2, label='Data 1' if i==0 else None)
    ax1.plot(bin_centers, h2_hist, color=orange_color, lw=2, label='Data 2' if i==0 else None)

ax1.set_xlabel('Value', fontsize=22)
ax1.set_ylabel('Density', fontsize=22)
ax1.set_title('Density Curves of Data 1 and Data 2', fontsize=22)
ax1.legend(fontsize=16)

# Create a second y-axis for the ratios plot
ax2 = ax1.twinx()
ax2.plot(means[:-10], ratios, marker='o', color='red', label='Max Ratio')
ax2.set_ylabel('Max Ratio', fontsize=22)
ax2.legend(fontsize=16, loc='upper left')

plt.savefig("./plots/bound/histogram_combined.pdf", format="pdf", bbox_inches="tight")
plt.show()

# plt.title('Log of the effective clipping ratio',fontsize=22)
# plt.savefig("./plots/bound/ratio.pdf", format="pdf", bbox_inches="tight")  

plt.show()



In [ ]:
import pandas as pd
import wandb
import os
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import itertools

sns.set(font_scale=1.3)
sns.set_theme()


os.environ["WANDB_API_KEY"]="28996bd59f1ba2c5a8c3f2cc23d8673c327ae230"
api = wandb.Api()
entity = "mahdikallel"
project_name = "PPO_OUTBOUND"
runs = api.runs(entity + "/" + project_name)

# Select one run from the list
run = runs[0]

# Retrieve the run's history containing the necessary metrics
history = run.history(samples=10000)
print(history.columns)


# Create a figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
# Plot both on the same axis using axes[0]
axes[0].plot(history["_step"], history["OOB_error"], marker='o', label="OOB_ERROR")
axes[0].plot(history["_step"], history["training/out_of_bound_percentage"]/100, marker='o', label="Out of Bound Percentage")
axes[0].set_title("OOB_ERROR & Out of Bound Percentage vs Step", fontsize=22)
axes[0].set_xlabel("Step", fontsize=16)
axes[0].set_ylabel("Value", fontsize=16)
axes[0].tick_params(labelsize=16)
axes[0].legend(fontsize=16)

# Remove the unused second subplot
fig.delaxes(axes[1])

plt.tight_layout()
plt.savefig("./plots/bound/error.pdf", format="pdf", bbox_inches="tight")  
plt.show()

